In [10]:

import pandas as pd
from io import StringIO



# Essayer différents séparateurs
df = pd.read_parquet("../data/bronze/communes/communes.parquet")
print(df.shape)
print(df.columns.tolist())
print(df.head())

(34935, 13)
['code_insee', 'nom_standard', 'epci_code', 'epci_nom', 'nom_unite_urbaine', 'code_postal', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'latitude_mairie', 'longitude_mairie', 'population']
  code_insee             nom_standard  epci_code                  epci_nom  \
0      01001  L'Abergement-Clémenciat  200069193           CC de la Dombes   
1      01002    L'Abergement-de-Varey  240100883  CC de la Plaine de l'Ain   
2      01004        Ambérieu-en-Bugey  240100883  CC de la Plaine de l'Ain   
3      01005      Ambérieux-en-Dombes  200042497    CC Dombes Saône Vallée   
4      01006                  Ambléon  200040350              CC Bugey Sud   

   nom_unite_urbaine  code_postal  reg_code               reg_nom dep_code  \
0                NaN       1400.0        84  Auvergne-Rhône-Alpes       01   
1                NaN       1640.0        84  Auvergne-Rhône-Alpes       01   
2  Ambérieu-en-Bugey       1500.0        84  Auvergne-Rhône-Alpes       01   
3               

In [11]:
# EXPLORATION DU PROBLÈME DE CORRESPONDANCE GÉOGRAPHIQUE
# ========================================================
# Les loyers sont par agglomération, les communes par unité urbaine
# On vérifie si les noms correspondent

import re

# Charger les deux fichiers
df_loyers = pd.read_parquet("../data/bronze/loyers/annee=2023/loyers_2023.parquet")
df_communes = pd.read_parquet("../data/bronze/communes/communes.parquet")

agglos_loyers = df_loyers["agglomeration"].unique()
unites_urbaines = df_communes["nom_unite_urbaine"].dropna().unique()

print(f"Agglomérations dans loyers OLAP : {len(agglos_loyers)}")
print(f"Unités urbaines dans fichier communes : {len(unites_urbaines)}")
print()

# Tentative de correspondance automatique
def extraire_ville(nom_agglo):
    nom = re.sub(r"Agglomération (de |d'|du |des |)", "", nom_agglo)
    return nom.strip()

matches_ok = []
matches_ko = []

for agglo in sorted(agglos_loyers):
    ville = extraire_ville(agglo)
    match = df_communes[
        df_communes["nom_unite_urbaine"].str.contains(ville, na=False, case=False)
    ]
    if len(match) > 0:
        matches_ok.append((agglo, ville, len(match)))
    else:
        matches_ko.append((agglo, ville))

print(f"✅ Correspondances trouvées automatiquement : {len(matches_ok)}/{len(agglos_loyers)}")
print(f"❌ Sans correspondance : {len(matches_ko)}/{len(agglos_loyers)}")
print()

print("--- Correspondances trouvées ---")
for agglo, ville, nb in matches_ok:
    print(f"  ✅ '{agglo}' → '{ville}' ({nb} communes)")

print()
print("--- Sans correspondance (nécessitent un mapping manuel) ---")
for agglo, ville in matches_ko:
    print(f"  ❌ '{agglo}' → '{ville}'")

Agglomérations dans loyers OLAP : 61
Unités urbaines dans fichier communes : 2458

✅ Correspondances trouvées automatiquement : 32/61
❌ Sans correspondance : 29/61

--- Correspondances trouvées ---
  ✅ 'Agglomération d'Ajaccio' → 'Ajaccio' (5 communes)
  ✅ 'Agglomération d'Alençon' → 'Alençon' (8 communes)
  ✅ 'Agglomération d'Alès' → 'Alès' (22 communes)
  ✅ 'Agglomération d'Annemasse' → 'Annemasse' (35 communes)
  ✅ 'Agglomération d'Arras' → 'Arras' (17 communes)
  ✅ 'Agglomération de Bastia' → 'Bastia' (7 communes)
  ✅ 'Agglomération de Besançon' → 'Besançon' (13 communes)
  ✅ 'Agglomération de Bordeaux' → 'Bordeaux' (73 communes)
  ✅ 'Agglomération de Brest' → 'Brest' (7 communes)
  ✅ 'Agglomération de Chalon-sur-Saône' → 'Chalon-sur-Saône' (13 communes)
  ✅ 'Agglomération de Châteauroux' → 'Châteauroux' (4 communes)
  ✅ 'Agglomération de Clermont-Ferrand' → 'Clermont-Ferrand' (17 communes)
  ✅ 'Agglomération de Draguignan' → 'Draguignan' (6 communes)
  ✅ 'Agglomération de Fréjus' 

C:\Users\ishimwe elodie\AppData\Local\Temp\ipykernel_14368\2225998266.py:30: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_communes["nom_unite_urbaine"].str.contains(ville, na=False, case=False)


In [ ]:
# CONCLUSION CORRESPONDANCE GÉOGRAPHIQUE
# =======================================
# 61 agglomérations dans les loyers OLAP
# 2458 unités urbaines dans le fichier communes
#
# Résultat de l'analyse :
#   ✅ 32/61 (52%) → correspondance automatique par nom
#   ❌ 5/61  (8%)  → DOM (La Réunion, Guadeloupe) → exclus du périmètre France métropolitaine
#   ❌ 14/61 (23%) → nom différent → mapping manuel nécessaire
#   ❌ 10/61 (17%) → zones rurales/atypiques → difficiles à mapper
#
# Solution retenue :
#   1. Correspondance automatique pour les 32 cas simples
#   2. Mapping manuel pour les 14 cas avec nom différent
#   3. Exclusion des DOM et zones rurales atypiques
#      → documenté comme limite du projet
#
# Couverture finale estimée : ~46/61 agglomérations (75%)